# Target acquisition

This is an example of using the ZMART Controller for a smart microscopy experiment. It is meant as a starting point to build a workflow further.

It assumes that the [ZMART Controller](https://github.com/ZMB-UZH/ZMART-controller) and the [ZMART driver](https://github.com/ZMB-UZH/ZMART-drivers) are installed properly, and that the microscope is ready to be controlled.

The vocabulary of the ZMART Controller is interoperable, so you can use it on different microscopes. You can also implement this with the documentation of the ZMART driver, but you would lose the interoperability.

Install what it needs first: see [the steps](README.md).

## 1. Connect

Discover the microscopes that are there and select the one for Leica.

In [ ]:
# The ZMART Controller: one way to talk to every microscope
from zmart_controller import mic

# List the microscopes configured on this computer
mic.get_instruments()


<br>

Connect to it. Here we connect to the mock microscope.

In [ ]:
# Connect to the mock microscope; its images are saved as OME-Zarr in the "images" folder
mic.connect("mock", {"output_root": "images", "format": "ome-zarr"})


<br>

## 2. Positions

Set up a list.

In [ ]:
# An empty list; each position you note down is added to it
positions = []


<br>

Move the stage to position 1 and note down the location.

In [ ]:
# Ask the microscope where the stage is now
xyz = mic.get_xyz()["output"]

# Add this position (x, y, z in micrometres) to the list
positions.append([xyz["x"]["position"], xyz["y"]["position"], xyz["z"]["position"]])


<br>

Move the stage to position 2 and note down the location.

In [ ]:
# Ask the microscope where the stage is now
xyz = mic.get_xyz()["output"]

# Add this position (x, y, z in micrometres) to the list
positions.append([xyz["x"]["position"], xyz["y"]["position"], xyz["z"]["position"]])


<br>

## 3. States

Set up a list.

In [ ]:
# An empty list; each state you note down is added to it
states = []


<br>

Set the microscope to the overview settings and note down the state.

In [ ]:
# Note down the overview settings (objective, laser, exposure, ...)
states.append(mic.get_state()["output"])


<br>

Set the microscope to the target settings and note down the state.

In [ ]:
# Note down the target settings
states.append(mic.get_state()["output"])


<br>

## 4. Analysis

Set up the analysis that finds the objects.

In [ ]:
# The ZMART analysis engine
from zmart_analysis import Engine

# The pipeline: finds nuclei with a watershed and measures each one
pipeline = "../../work/ZMART-analysis/workflows/object_analysis/pipelines/object_analysis_watershed.yaml"

# Start the analysis
analysis = Engine()

# Name the pipeline "detect_targets"; its results go to the "analysis_output" folder
analysis.register("detect_targets", pipeline, "analysis_output")


<br>

Set up a list for the analysis jobs.

In [ ]:
# An empty list; each overview image sent to the analysis adds its job to it
jobs = []


<br>

## 5. Acquire targets

Image the overview positions, find the dividing cells, then image them.

In [ ]:
# To wait a moment while the analysis runs
import time

# How many objects to image per overview
N_TARGETS = 1

# 1. Image and analyse each position
for i, (x, y, z) in enumerate(positions):
    # Move the stage
    mic.set_xyz(x, y, z)

    # Switch the state
    mic.set_state(states[0])

    # Acquire an image
    image = mic.acquire(position_label=f"{i}", acquisition_type="overview")["output"]

    # Submit for analysis
    jobs.append(analysis.submit("detect_targets", {"image_path": image["files"][0]}))


# 2. Wait for the analysis
for job in jobs:
    while analysis.status("detect_targets")[job]["state"] not in ("done", "failed"):
        time.sleep(0.2)


# 3. Choose targets
targets = []
for i, job in enumerate(jobs):
    # Read the objects found
    table = analysis.status("detect_targets")[job]["output"]["object_analysis"]["objects"]["properties"]

    # Score each object: high intensity CV and high eccentricity is often a cell in mitosis
    objects = []
    for cv, eccentricity, x, y in zip(table["intensity_cv"], table["eccentricity"], table["stage_x_um"], table["stage_y_um"]):
        objects.append((cv * eccentricity, x, y))

    # Sort them, best first
    objects.sort(reverse=True)

    # Keep the best
    for score, x, y in objects[:N_TARGETS]:
        targets.append([x, y, positions[i][2]])


# 4. Image each target
for n, (x, y, z) in enumerate(targets):
    # Move the stage to the object
    mic.set_xyz(x, y, z)

    # Switch the state
    mic.set_state(states[1])

    # Acquire an image
    mic.acquire(position_label=f"{n}", acquisition_type="target")


<br>

Close the connections.

In [ ]:
# Stop the analysis
analysis.shutdown()

# Close the connection to the microscope
mic.disconnect()
